In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [ ]:
import Data_import

In [ ]:
PCDM_DEMOGRAPHIC = Data_import.PCDM_DEMOGRAPHIC
PCDM_DEATH_CAUSE = Data_import.PCDM_DEATH_CAUSE
PCDM_DEATH = Data_import.PCDM_DEATH
PCDM_DIAGNOSIS = Data_import.PCDM_DIAGNOSIS
PCDM_ENCOUNTER = Data_import.PCDM_ENCOUNTER
PCDM_IMMUNIZATION = Data_import.PCDM_IMMUNIZATION
PCDM_LAB_RESULT_CM = Data_import.PCDM_LAB_RESULT_CM
PCDM_LDS_ADDRESS_HISTORY = Data_import.PCDM_LDS_ADDRESS_HISTORY
PCDM_MED_ADMIN = Data_import.PCDM_MED_ADMIN
PCDM_OBS_CLIN = Data_import.PCDM_OBS_CLIN
PCDM_OBS_GEN = Data_import.PCDM_OBS_GEN
PCDM_PRESCRIBING = Data_import.PCDM_PRESCRIBING
PCDM_PROCEDURES = Data_import.PCDM_PROCEDURES
PCDM_PROVIDER = Data_import.PCDM_PROVIDER
PCDM_VITAL = Data_import.PCDM_VITAL

In [ ]:
### Translation dictionary for diagnosis codes
DiagCodeTranslationDF = pd.read_csv('ICD_RX translation/HNC Diagnosis ICD Translation DF.csv')
diagTranslateDict=dict(zip(DiagCodeTranslationDF['ICD_CODES'], DiagCodeTranslationDF['DESCRIPTION']))

### Translation dictionary for procedures
ProcedCodeTranslationDf = pd.read_csv('ICD_RX translation/HNC procedure ICD Translation DF.csv')
procedTranlateDict=dict(zip(ProcedCodeTranslationDf['ICD_CODES'], ProcedCodeTranslationDf['DESCRIPTION']))

### Translation dictionary for rxnorm cui codes
column_types = {'RX_CODE':str, 'TRANSLATION':str}
rxCodeTranslatedf = pd.read_csv('ICD_RX translation/rxCodeTranslationDF_HNC.csv', dtype=column_types)
rxCodeTranslatedf['TRANSLATION'] = rxCodeTranslatedf['TRANSLATION'].fillna('')
rxCodeTranslateDict =dict(zip(rxCodeTranslatedf['RX_CODE'], rxCodeTranslatedf['TRANSLATION']))

### Translation dictionary for NDC med codes
column_types = {'NDC_CODE':str, 'TRANSLATION':str}
ndcTranslatedf = pd.read_csv('ICD_RX translation/ndcCodeTranslationDF_HNC.csv', dtype=column_types)
ndcTranslatedf['TRANSLATION'] = ndcTranslatedf['TRANSLATION'].fillna('')
ndcCodeTranslateDict =dict(zip(ndcTranslatedf['NDC_CODE'], ndcTranslatedf['TRANSLATION']))


### PCDM_DIAGNOSIS/ PCDM_DEMOGRAPHIC

#### Patient data

To extract key patient information:

- age at HNC diagnosis
- date of HNC diagnosis
- gender
- metastasis status/ date of metastasis diagnosis

to extract diagnosis codes:

- diagnosis codes
- date of diagnosis

##### AGE at diagnosis/ HNC date of diagnosis

In [ ]:
list(PCDM_DIAGNOSIS['PATID'].unique())

In [ ]:
PCDM_DEMOGRAPHIC.head()

In [ ]:
pat_gender = {}
for pat in tqdm(PCDM_DEMOGRAPHIC['PATID'].unique()):
    if PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['SEX'].values[0] == 'M':
        pat_gender[pat] = 1
    else:
        pat_gender[pat] = 0


In [ ]:
pd.DataFrame(columns = ['PCDM_PATID', 'AGE_AT_DIAGNOSIS', 'DATE_OF_HNC_DIAGNOSIS', 'GENDER', 'METASTASIS_STATUS', 'DATE_OF_METASTASIS'])
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].isnull()]
### Get age at diagnosis
### identify date of HNC diagnosis
HNC_ICD_9 = [140, 141, 142, 143, 144, 145, 146, 147, 148, 149]
HCN_ICD_10 = ['C00', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C31', 'C80.1','C76', 'C44.42', 'C30', 'C31', 'C32', 'C33']

ICD_9_diag = []
for val in PCDM_DIAGNOSIS['DX']:
    for val2 in HNC_ICD_9:
        if str(val2) in str(val):
            #print(val)
            ICD_9_diag.append(val)

ICD_10_diag = []
for val in PCDM_DIAGNOSIS['DX']:
    for val2 in HCN_ICD_10:
        if str(val2) in str(val):
            #print(val)
            ICD_10_diag.append(val)

HNC_DIAG = PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['DX'].isin(ICD_9_diag + ICD_10_diag)]
HNC_DIAG_PAT = {}
for pat in tqdm(PCDM_DIAGNOSIS['PATID'].unique()):
    # print(pat)
    # print(HNC_DIAG[HNC_DIAG['PATID']==pat]['DX'])
    # print(HNC_DIAG[HNC_DIAG['PATID']==pat]['DX_DATE'])
    # print(HNC_DIAG[HNC_DIAG['PATID']==pat]['DX_DATE'].min())
    HNC_DIAG_PAT[pat] = HNC_DIAG[HNC_DIAG['PATID']==pat]['DX_DATE'].min()

hnc_diagnosis_date_data = pd.DataFrame.from_dict(HNC_DIAG_PAT, orient = 'index',columns=['HNC_DIAG_DATE']).reset_index(names='PATID')

age_at_diagnosis = {}
for pat in tqdm(PCDM_DIAGNOSIS['PATID'].unique()):
    # print(pat)
    # print(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PCDM_PATID']==pat]['BIRTH_DATE'])
    # print(HNC_DIAG_PAT[pat])
    # print(pat)
    # print('diag')
    # print(pd.to_datetime(HNC_DIAG_PAT[pat]))
    # print('demo')
    # print(pd.to_datetime(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['BIRTH_DATE']))
          
    # print((pd.to_datetime(HNC_DIAG_PAT[pat])- pd.to_datetime(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['BIRTH_DATE'])).dt.days/365.25)
    try:
        age_at_diagnosis[pat] =  int( (pd.to_datetime(HNC_DIAG_PAT[pat])- pd.to_datetime(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['BIRTH_DATE'])).dt.days/365.25 )
    except:
        age_at_diagnosis[pat] = np.nan 

age_data=pd.DataFrame.from_dict(age_at_diagnosis, orient = 'index',columns=['AGE_AT_DIAGNOSIS']).reset_index(names='PATID')

pat_gender = {}
for pat in tqdm(PCDM_DEMOGRAPHIC['PATID'].unique()):
    if PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['SEX'].values[0] == 'M':
        pat_gender[pat] = 1
    else:
        pat_gender[pat] = 0

pat_gender_data = pd.DataFrame.from_dict(pat_gender, orient = 'index',columns=['GENDER']).reset_index(names='PATID')


##### Metastasis status

In [ ]:
### Get list of dates for patients
commonPats = list(set(PCDM_DIAGNOSIS['PATID'])
                   .intersection(PCDM_MED_ADMIN['PATID'])
                   .intersection(PCDM_PROCEDURES['PATID'])
                   .intersection(PCDM_PRESCRIBING['PATID'])
                )
pat_outcome_dates = {}
for pat in tqdm(commonPats):
    dates = []
    dates.extend(PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['PATID']==pat]['DX_DATE'].tolist())
    dates.extend(PCDM_MED_ADMIN[PCDM_MED_ADMIN['PATID']==pat]['MEDADMIN_START_DATE'].tolist())
    dates.extend(PCDM_PROCEDURES[PCDM_PROCEDURES['PATID']==pat]['PX_DATE'].tolist())
    dates.extend(PCDM_PRESCRIBING[PCDM_PRESCRIBING['PATID']==pat]['RX_START_DATE'].tolist())
    dates = [x for x in dates if type(x) != float]
    dates = [x for x in dates if x is not None]
    for date in dates:
        try:
            pd.to_datetime(date)
        except:
            dates.remove(date)
    # dates = [pd.to_datetime(x) for x in dates]
    pat_outcome_dates[pat] = max(dates)


In [ ]:
print(pat)

In [ ]:
len(PCDM_DIAGNOSIS)

In [ ]:
# Remove rows with invalid PATID entries
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].isnull()]
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].str.contains('à', na=False)]
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].str.contains('è', na=False)]
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].str.contains('nan', na=False)]
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].str.contains('None', na=False)]
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].str.contains(' ', na=False)]
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].str.contains('  ', na=False)]
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS[~PCDM_DIAGNOSIS['PATID'].str.contains("\x00", na=False)]

In [ ]:
len(PCDM_DIAGNOSIS)

In [ ]:
metastasisICD = []
for val in tqdm(diagTranslateDict.keys()):
    #print(str.lower(str(translateDict[val])))
    if 'metas' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'spread' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary unspecified malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary and unspecified malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)

In [ ]:
len(metastasisICD)

In [ ]:
metastasisICD = []
for val in tqdm(diagTranslateDict.keys()):
    #print(str.lower(str(translateDict[val])))
    if 'metas' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'spread' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary unspecified malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary and unspecified malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)

### Get metastasis status
onlyMetData = PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['DX'].isin(metastasisICD)]
### Create list of patients with metastasis ICD codes
metPatients=list(set(onlyMetData['PATID']))

### Get date of metastasis
patid = []
outcome = []
outDate = []
for pat in tqdm(PCDM_DIAGNOSIS['PATID'].unique()):
    patid.append(pat)
    if pat in metPatients:
        outDate.append( onlyMetData[onlyMetData['PATID']==pat]['DX_DATE'].min() )
        outcome.append(1)
    else:
        try:
            outDate.append(pat_outcome_dates[pat])
            outcome.append(0)
        except:
            outDate.append(PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['PATID']==pat]['DX_DATE'].max())
            outcome.append(0)

outcomeData = {
    'PATID': patid,
    'METASTASIS': outcome,
    'OUTCOME_DATE': outDate
}       
metastasis_data = pd.DataFrame(outcomeData)

print(len(set(metastasis_data[metastasis_data['METASTASIS']==1]['PATID'])))
print(len(set(metastasis_data[metastasis_data['METASTASIS']==0]['PATID'])))
print(len(set(metastasis_data[metastasis_data['METASTASIS']==1]['PATID']))/ len(set(metastasis_data[metastasis_data['METASTASIS']==0]['PATID'])))

In [ ]:
print(pat)

In [ ]:
print(len(set(metastasis_data['PATID'])))

### Merge it all

#### Patient level data

In [ ]:
hnc_diagnosis_date_data.head()

In [ ]:
age_data.head()

In [ ]:
pat_gender_data.head()

In [ ]:
metastasis_data.head()

In [ ]:
# hnc_diagnosis_date_data = hnc_diagnosis_date_data[hnc_diagnosis_date_data['PATID'].isin(commonPats)]
# age_data = age_data[age_data['PATID'].isin(commonPats)]
# pat_gender_data = pat_gender_data[pat_gender_data['PATID'].isin(commonPats)]
# metastasis_data = metastasis_data[metastasis_data['PATID'].isin(commonPats)]

# patient_data = hnc_diagnosis_date_data.merge(age_data, on = ['PATID'], how = 'outer')
# patient_data = patient_data.merge(pat_gender_data, on = ['PATID'], how = 'outer')
# patient_data = patient_data.merge(metastasis_data, on = ['PATID'], how = 'outer')

##### Merge it all to one patient level data frame

In [ ]:
hnc_diagnosis_date_data = hnc_diagnosis_date_data.set_index('PATID')
age_data = age_data.set_index('PATID')
pat_gender_data = pat_gender_data.set_index('PATID')
metastasis_data = metastasis_data.set_index('PATID')

patient_data = hnc_diagnosis_date_data.join(age_data, how = 'outer')
patient_data = patient_data.join(pat_gender_data, how = 'outer')
patient_data = patient_data.join(metastasis_data, how = 'outer')
patient_data = patient_data.reset_index()

In [ ]:
patient_data = patient_data.reset_index()

In [ ]:
patient_data = patient_data[~patient_data['PATID'].isnull()]
patient_data = patient_data[~patient_data['PATID'].str.contains('à')]
patient_data = patient_data[~patient_data['PATID'].str.contains('nan')]
patient_data = patient_data[~patient_data['PATID'].str.contains('None')]
patient_data = patient_data[~patient_data['PATID'].str.contains(' ')]
patient_data = patient_data[~patient_data['PATID'].str.contains('  ')]
patient_data = patient_data[~patient_data['PATID'].str.contains("\x00")]
patient_data = patient_data[~patient_data['HNC_DIAG_DATE'].isnull()]

In [ ]:
patient_data[patient_data['METASTASIS']==1]

In [ ]:
patient_data[patient_data['METASTASIS']==0]

In [ ]:
# Check if any metastasis date is before HNC diagnosis date
sanity_check = patient_data[patient_data['OUTCOME_DATE'] < patient_data['HNC_DIAG_DATE']]
print(f"Number of patients with metastasis before diagnosis: {len(sanity_check)}")
sanity_check[['PATID', 'HNC_DIAG_DATE', 'OUTCOME_DATE', 'METASTASIS']]

In [ ]:
len(patient_data[patient_data['METASTASIS']==1])/len(patient_data)

In [ ]:
patient_data.to_csv('input_data/patient level data.csv')

### Demographics table

In [ ]:
PCDM_DEMOGRAPHIC

In [ ]:
patient_data

In [ ]:
# Add an age group column to the patient_data dataframe
patient_data['AGE_GROUP'] = pd.cut(patient_data['AGE_AT_DIAGNOSIS'], 
                                   bins=range(0, 101, 10), 
                                   labels=[f"{i}-{i+9}" for i in range(0, 100, 10)], 
                                   right=False)

# Group by age group and count the number of patients in each group
age_group_table = patient_data.groupby('AGE_GROUP').size().reset_index(name='PATIENT_COUNT')

print(age_group_table)

In [ ]:
age_group_table

In [ ]:
# Group by gender and count the number of patients in each group
gender_table = patient_data.groupby('GENDER').size().reset_index(name='PATIENT_COUNT')

print(gender_table)

In [ ]:
gender_table

In [ ]:
gender_table['Percentage'] = gender_table['PATIENT_COUNT']/np.sum(gender_table['PATIENT_COUNT']) *100

In [ ]:
gender_table

In [ ]:
age_group_table['Percentage'] = age_group_table['PATIENT_COUNT']/np.sum(age_group_table['PATIENT_COUNT']) *100

In [ ]:
age_group_table

In [ ]:
patient_data